# Creating AI without such libraries as Torch or TensorFlow

## Load dataset and STOI

In [ ]:
import numpy as np
import json

data = np.memmap("train.bin", dtype=np.uint16, mode="r")

with open("stoi.json", "r") as f:
    stoi = json.load(f)

print(f"Vocabulary length: {len(stoi)}")
print(f"Tokens length: {len(data)}")

Vocabulary length: 194263
Tokens length: 100000404


## Split dataset into training and test

In [2]:
data = data

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [3]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else test_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

## Traing loop

In [ ]:
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 128
vocabulary_size = len(stoi)
block_size = 64
batch_size = 16
block_layers = 4
sgd = Adam(lr=3e-4)

model = MiniGPT(vocab_size=vocabulary_size, d_model=d_model, block_size=block_size, n_layers=block_layers, gradient=sgd)
xb, yb = get_batch("train", block_size, batch_size)

for step in range(5000):
    xb, yb = get_batch("train", block_size=64, batch_size=32)

    logits, loss = model.forward(xb, yb)
    
    model.backward()

    if step % 50 == 0:
        print("step:", step, "loss:", loss)

In [ ]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        
        logits, _ = model.forward(idx_cond, np.array([1]))
        
        logits = logits[:, -1, :]
        
        max_logits = np.max(logits, axis=-1, keepdims=True)
        exp_logits = np.exp(logits - max_logits)
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        next_tokens = []
        for b in range(probs.shape[0]):
            next_token = np.random.choice(len(vocabulary), p=probs[b])
            next_tokens.append(next_token)
        
        next_token = np.array(next_tokens).reshape(-1, 1)
        
        idx = np.concatenate([idx, next_token], axis=1)
    
    return idx

In [ ]:
itos = {i: ch for ch, i in stoi.items()}

prompt = "Hello"
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.int64)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 100)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

Hellog, SwiftUb app belops it befor the rest of the codebase for project is an butten type All, perhoning
